# 12 - Escalation Prediction
Predict complaint escalation using NLP features + ML.

In [1]:
import pandas as pd, numpy as np, joblib, os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, classification_report
import plotly.express as px, plotly.graph_objects as go
import warnings; warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED)
PRIMARY='#635BFF'; RISK='#E74C3C'; SAFE='#27AE60'; NEUTRAL='#3498DB'; WARNING='#F39C12'; TEMPLATE='plotly_white'


In [2]:
df = pd.read_csv('data/processed/complaints_with_nlp.csv')
print(f"Shape: {df.shape}")

# Target engineering
untimely_responses = ['Untimely response','In progress','No response',"Company can't substantiate response"]
high_emotion = ['Anger','Legal Threat','Distress']

df['escalation_flag'] = (
    (df['company_response'].isin(untimely_responses)) |
    (df['timely_response'] == 'No') |
    (df['emotion'].isin(high_emotion))
).astype(int)

print(f"Escalation rate: {df['escalation_flag'].mean():.4f}")
print(df['escalation_flag'].value_counts())


Shape: (500, 20)
Escalation rate: 0.6080
escalation_flag
1    304
0    196
Name: count, dtype: int64


In [3]:
# Feature engineering
le_cat = LabelEncoder()
le_emo = LabelEncoder()
le_prod = LabelEncoder()
le_via = LabelEncoder()

df['category_encoded'] = le_cat.fit_transform(df['complaint_category'].fillna('Unknown'))
df['emotion_encoded'] = le_emo.fit_transform(df['emotion'].fillna('Neutral'))
df['product_encoded'] = le_prod.fit_transform(df['Product'].fillna('Unknown'))
df['via_encoded'] = le_via.fit_transform(df['submitted_via'].fillna('Unknown'))
df['timely_binary'] = (df['timely_response'] == 'No').astype(int)
df['narrative_length'] = df['narrative'].apply(lambda x: len(str(x).split()))

feature_cols = ['category_encoded','emotion_encoded','product_encoded','via_encoded','timely_binary','narrative_length']
X = df[feature_cols]
y = df['escalation_flag']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
ratio = (y_train==0).sum() / max((y_train==1).sum(), 1)


Train: (400, 6), Test: (100, 6)


In [4]:
# Train models
rf = RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=SEED)
rf.fit(X_train, y_train)

xgb = XGBClassifier(scale_pos_weight=ratio, eval_metric='auc', random_state=SEED, use_label_encoder=False)
xgb.fit(X_train, y_train)

for name, model in [('Random Forest', rf), ('XGBoost', xgb)]:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    print(f"\n{name}:")
    print(f"  Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"  F1: {f1_score(y_test, y_pred):.4f}")
    print(f"  ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")



Random Forest:
  Accuracy: 0.9900
  F1: 0.9919
  ROC-AUC: 1.0000

XGBoost:
  Accuracy: 1.0000
  F1: 1.0000
  ROC-AUC: 1.0000


In [5]:
# Generate escalation probability
df['escalation_probability'] = xgb.predict_proba(X)[:,1]


In [6]:
# Plot a: Escalation rate overall
esc = df['escalation_flag'].value_counts().reset_index(); esc.columns=['flag','count']
esc['label'] = esc['flag'].map({0:'No Escalation',1:'Escalation'})
fig = px.pie(esc, values='count', names='label', hole=0.3,
             color_discrete_sequence=[SAFE, RISK], template=TEMPLATE, title='Escalation Rate')
fig.show()


In [7]:
# Plot b: Escalation rate by category
esc_cat = df.groupby('complaint_category')['escalation_flag'].mean().sort_values(ascending=False).reset_index()
fig = px.bar(esc_cat, x='complaint_category', y='escalation_flag', color_discrete_sequence=[RISK],
             template=TEMPLATE, title='Escalation Rate by Category')
fig.update_xaxes(tickangle=45)
fig.show()


In [8]:
# Plot c: Escalation rate by emotion
esc_emo = df.groupby('emotion')['escalation_flag'].mean().sort_values(ascending=False).reset_index()
fig = px.bar(esc_emo, x='emotion', y='escalation_flag', color_discrete_sequence=[WARNING],
             template=TEMPLATE, title='Escalation Rate by Emotion')
fig.show()


In [9]:
# Plot d: Escalation probability histogram
fig = px.histogram(df, x='escalation_probability', nbins=30, color_discrete_sequence=[RISK],
                   template=TEMPLATE, title='Escalation Probability Distribution')
fig.show()


In [10]:
# Plot e: ROC curves
fig = go.Figure()
for name, model, color in [('Random Forest', rf, PRIMARY), ('XGBoost', xgb, RISK)]:
    y_proba = model.predict_proba(X_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{name} (AUC={auc:.3f})", line=dict(color=color)))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], name='Random', line=dict(dash='dash', color='gray')))
fig.update_layout(title='ROC Curves', xaxis_title='FPR', yaxis_title='TPR', template=TEMPLATE)
fig.show()


In [11]:
# Plot f: Feature importance RF
imp = pd.DataFrame({'feature': feature_cols, 'importance': rf.feature_importances_})
imp = imp.sort_values('importance', ascending=True)
fig = px.bar(imp, y='feature', x='importance', orientation='h',
             color_discrete_sequence=[PRIMARY], template=TEMPLATE, title='Feature Importance - RF')
fig.show()


In [12]:
# Plot g: Escalation by Product top 10
esc_prod = df.groupby('Product')['escalation_flag'].mean().sort_values(ascending=False).head(10).reset_index()
fig = px.bar(esc_prod, x='Product', y='escalation_flag', color_discrete_sequence=[WARNING],
             template=TEMPLATE, title='Escalation Rate by Product (Top 10)')
fig.update_xaxes(tickangle=45)
fig.show()


In [13]:
# Save
os.makedirs('models/escalation', exist_ok=True)
joblib.dump(xgb, 'models/escalation/xgboost_escalation.pkl')
print("Saved escalation model")

df.to_csv('data/processed/complaints_with_escalation.csv', index=False)
print(f"Saved complaints_with_escalation.csv - shape: {df.shape}")


Saved escalation model
Saved complaints_with_escalation.csv - shape: (500, 27)
